# 01 Introduction to Enterprise RAG Architecture

**Real-World Scenario**: Global Fintech Microservice SLA & Tech Spec Search System.

This notebook demonstrates constructing an enterprise baseline RAG pipeline using LangChain `ChatOpenAI`, saving vector stores inside a hidden `.vectordb/` directory, and executing grounded QA.

## Step 1: Environment Setup & API Key Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Realistic Dataset & Problem Domain Setup

We define technical architecture specifications for microservices, SLAs, and security tokens.

In [ ]:
tech_specs = [
    "Microservice Architecture: Microservice A communicates with Database B via gRPC over TLS 1.3 on port 50051.",
    "Latency SLA: API endpoints must maintain a P99 latency of under 100 milliseconds and P95 under 50ms.",
    "Security Policy: All JWT authentication tokens expire after 3600 seconds (1 hour) and must use RS256 signing.",
    "Database Failover: PostgreSQL primary database replica failover occurs automatically after 3 missed healthcheck heartbeats (15s)."
]


## Step 3: Indexing Specs into FAISS Vector DB in Hidden `.vectordb/` Directory

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

if os.getenv("OPENAI_API_KEY"):
    embeddings = OpenAIEmbeddings()
    # Create FAISS index
    vectorstore = FAISS.from_texts(tech_specs, embeddings)
    
    # Save index into hidden directory to prevent git tracking
    db_save_path = os.path.join(".vectordb", "01_faiss_index")
    vectorstore.save_local(db_save_path)
    print(f"=== FAISS Vector Index Successfully Saved to Hidden Folder: {db_save_path} ===")


## Step 4: Loading Hidden Vector DB & Executing Grounded RAG Query

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    db_save_path = os.path.join(".vectordb", "01_faiss_index")
    loaded_vectorstore = FAISS.load_local(db_save_path, embeddings, allow_dangerous_deserialization=True)
    retriever = loaded_vectorstore.as_retriever(search_kwargs={"k": 2})
    
    prompt = ChatPromptTemplate.from_template('''Answer question strictly using context:
Context:
{context}

Question: {question}
Answer:''')
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    
    query = "What is the communication protocol and port between Microservice A and Database B?"
    docs = retriever.invoke(query)
    context_text = "\n".join([d.page_content for d in docs])
    
    response = llm.invoke(prompt.format(context=context_text, question=query))
    print("=== Baseline Enterprise RAG Query Output ===")
    print("Retrieved Chunks:")
    for idx, d in enumerate(docs, 1):
        print(f"  [{idx}] {d.page_content}")
    print("\nLLM Response:")
    print(response.content)
